In [3]:
library(data.table)
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::between()     masks data.table::between()
✖ dplyr::filter()      masks stats::filter()
✖ dplyr::first()       masks data.table::first()
✖ lubridate::hour()    masks data.table::hour()
✖ lubridate::isoweek() masks data.table::isoweek()
✖ dplyr::lag()         masks stats::lag()
✖ dplyr::last()        masks data.table::last()
✖ lubridate::mday()    masks data.table::mday()
✖ lubridate::minute()  masks data.table::minute()
✖ lubridate::month()   masks data.table::month()
✖ lubridate::quarter() masks data.table::quarter()
✖ lubridate::second()  masks data.table::second()
✖ purrr::transpose()   masks data.table::transpose()
✖ lubridate::wday() 

In [ ]:
celltype1 <- "KELLYspecific-KELLY"
celltype2 <- "all"
output_filename <- "/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_denovo_KELLY_PSI/WT_KELLYspecific-KELLY_all/KELLYspecific-KELLY_all_formatted_df.tsv"

print(paste("Celltype 1: ", celltype1))
print(paste("Celltype 2: ", celltype2))
print(paste("Output filename: ", output_filename))
all_sample_reps <- fread("/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_denovoR2/denovoR2_all_samples_PSI_count_table.csv")
all_sample_reps <- all_sample_reps %>% 
  filter(grepl("KELLYspecific", condition)) %>% 
  mutate(index_offset = paste0(index_offset, "__", design)) 
print("Finished reading data")

get_paradise_output <- function(df, condition1, condition2) {
  # Filter data frames based on conditions
  condition1_df <- df %>% filter(condition == condition1) %>% mutate(PSI = INCLUDED / (INCLUDED + SKIPPED))
  if (condition2 == "all") {
    condition2_df <- df %>% filter(condition != condition1) %>% mutate(PSI = INCLUDED / (INCLUDED + SKIPPED))
  } else {
    condition2_df <- df %>% filter(condition == condition2) %>% mutate(PSI = INCLUDED / (INCLUDED + SKIPPED))
  }
  
  condition1_df_PSI <- condition1_df %>% group_by(index_offset) %>% 
    summarise(PSI_1 = mean(PSI,na.rm=T))

  condition2_df_PSI <- condition2_df %>% group_by(index_offset) %>% 
    summarise(PSI_2 = mean(PSI, na.rm = T))


  # Merge the 2 dataframe.
  # Also filter out the indices with PSI difference less than 0.1 so that we don't have to run calculations for them.
  condition1_2_merged <- merge(condition1_df_PSI, condition2_df_PSI, by = "index_offset")  %>% 
                            mutate(PSI_diff = abs(PSI_1 - PSI_2)) %>% filter(PSI_diff >= 0.1)
     
  print(head(condition1_2_merged))
  # Create a list of all unique indices.
  unique_indices <- unique(condition1_2_merged$index_offset)
  
  # Function to pad vectors with zeros
  pad_with_zeros <- function(vec, target_len = 2) {
    if (length(vec) < target_len) {
      vec <- c(vec, rep(0, target_len - length(vec)))
    }
    return(vec)
  }
  
  
  # Create the paradise data frame using dplyr operations and future.apply for parallel processing
  paradise_df <- lapply(unique_indices, function(idx) {
    condition1_index_df <- condition1_df %>% filter(index_offset == idx) %>% arrange(sample)
    condition2_index_df <- condition2_df %>% filter(index_offset == idx) %>% arrange(sample)
    
    I1 <- pad_with_zeros(condition1_index_df$included_count)
    S1 <- pad_with_zeros(condition1_index_df$skipped_count)
    I2 <- pad_with_zeros(condition2_index_df$included_count)
    S2 <- pad_with_zeros(condition2_index_df$skipped_count)
    
    return (data.frame(
      ExonID = idx, 
      I1 = paste(I1, collapse = ","), 
      S1 = paste(S1, collapse = ","), 
      I2 = paste(I2, collapse = ","), 
      S2 = paste(S2, collapse = ","), 
      I_len = 1, 
      S_len = 1
    ))
  }) %>% bind_rows()
  
  return(paradise_df)
}

formatted_df <- get_paradise_output(all_sample_reps, celltype1, celltype2)

[1] "Celltype 1:  KELLYspecific-KELLY"
[1] "Celltype 2:  all"
[1] "Output filename:  /mnt/dawnccle2/processed_data/reprocess_250221/pairadise_denovo_KELLY_PSI/WT_KELLYspecific-KELLY_all/KELLYspecific-KELLY_all_formatted_df.tsv"


[1] "Finished reading data"
                                                                                                                    index_offset
1        ENSG00000004534.15;RBM6;chr3-50068689-50068764-50066241-50066502-50075200-50075330__0:0:0__vae_Kelly_1_to_pos_R2design8
2        ENSG00000004534.15;RBM6;chr3-50068689-50068764-50066241-50066502-50075200-50075330__0:0:0__vae_Kelly_1_to_pos_R2design9
3 ENSG00000006607.14;FARP2;chr2-241462522-241462612-241436480-241436538-241463334-241463458__0:0:0__vae_Kelly_1_to_pos_R2design1
4 ENSG00000006607.14;FARP2;chr2-241462522-241462612-241436480-241436538-241463334-241463458__0:0:0__vae_Kelly_1_to_pos_R2design2
5 ENSG00000006607.14;FARP2;chr2-241462522-241462612-241436480-241436538-241463334-241463458__0:0:0__vae_Kelly_1_to_pos_R2design3
6 ENSG00000006607.14;FARP2;chr2-241462522-241462612-241436480-241436538-241463334-241463458__0:0:0__vae_Kelly_1_to_pos_R2design4
      PSI_1      PSI_2  PSI_diff
1 0.1745216 0.01368818 0.1608334
2 0

In [14]:
library(tidyverse)
library(vroom)
library(data.table)
library(future)
library(future.apply)

# if pairadise is not installed, install it
if(!requireNamespace("PAIRADISE", quietly = TRUE)) {
  BiocManager::install("PAIRADISE")
}
# Also for BiocParallel
if(!requireNamespace("BiocParallel", quietly = TRUE)) {
  BiocManager::install("BiocParallel")
}
library(PAIRADISE)

reverse_complement <- function(dna_seq) {
  complement <- c("A" = "T", "T" = "A", "C" = "G", "G" = "C")
  nucleotides <- unlist(strsplit(dna_seq, ""))
  complement_nucleotides <- complement[nucleotides]
  reverse_complement_seq <- paste(rev(complement_nucleotides), collapse = "")
  return(reverse_complement_seq)
}

args <- commandArgs(trailingOnly = TRUE)
celltype1 <- "CANCERspecific-CH3-1"
celltype2 <- "CANCERspecific-FUBP1"
output_filename <-"/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_denovo_CANCER_PSI/WT_CANCERspecific-CH3-1_CANCERspecific-FUBP1/CANCERspecific-CH3-1_CANCERspecific-FUBP1_formatted_df.tsv"

print(paste("Celltype 1: ", celltype1))
print(paste("Celltype 2: ", celltype2))
print(paste("Output filename: ", output_filename))
all_sample_reps <- fread("/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_denovoR2/denovoR2_all_samples_PSI_count_table.csv")
all_sample_reps <- all_sample_reps %>% 
  filter(grepl("CANCERspecific", condition)) %>% 
  mutate(index_offset = paste0(index_offset, "__", design)) 
print("Finished reading data")


get_paradise_output <- function(df, condition1, condition2) {
  # Filter data frames based on conditions
  condition1_df <- df %>% filter(grepl(condition1, condition)) %>% mutate(PSI = INCLUDED / (INCLUDED + SKIPPED))
  if (condition2 == "all") {
    condition2_df <- df %>% filter(!grepl(condition1, condition)) %>% mutate(PSI = INCLUDED / (INCLUDED + SKIPPED))
  } else {
    condition2_df <- df %>% filter(grepl(condition2, condition)) %>% mutate(PSI = INCLUDED / (INCLUDED + SKIPPED))
  }

  print(unique(condition1_df$sample))
  print(unique(condition2_df$sample))
  
  condition1_df_PSI <- condition1_df %>% group_by(index_offset) %>% 
    summarise(PSI_1 = mean(PSI,na.rm=T))

  condition2_df_PSI <- condition2_df %>% group_by(index_offset) %>% 
    summarise(PSI_2 = mean(PSI, na.rm = T))

  # Merge the 2 dataframe.
  # Also filter out the indices with PSI difference less than 0.1 so that we don't have to run calculations for them.
  condition1_2_merged <- merge(condition1_df_PSI, condition2_df_PSI, by = "index_offset")  %>% 
                            mutate(PSI_diff = abs(PSI_1 - PSI_2)) # %>% filter(PSI_diff >= 0.1)

  # Create a list of all unique indices.
  unique_indices <- unique(condition1_2_merged$index_offset)
  
  # Function to pad vectors with zeros
  pad_with_zeros <- function(vec, target_len = 2) {
    if (length(vec) < target_len) {
      vec <- c(vec, rep(0, target_len - length(vec)))
    }
    return(vec)
  }
  
  
  # Create the paradise data frame using dplyr operations and future.apply for parallel processing
  paradise_df <- lapply(unique_indices, function(idx) {
    condition1_index_df <- condition1_df %>% filter(index_offset == idx) %>% arrange(sample)
    condition2_index_df <- condition2_df %>% filter(index_offset == idx) %>% arrange(sample)
    
    I1 <- pad_with_zeros(condition1_index_df$INCLUDED)
    S1 <- pad_with_zeros(condition1_index_df$SKIPPED)
    I2 <- pad_with_zeros(condition2_index_df$INCLUDED)
    S2 <- pad_with_zeros(condition2_index_df$SKIPPED)
    
    return (data.frame(
      ExonID = idx, 
      I1 = paste(I1, collapse = ","), 
      S1 = paste(S1, collapse = ","), 
      I2 = paste(I2, collapse = ","), 
      S2 = paste(S2, collapse = ","), 
      I_len = 1, 
      S_len = 1
    ))
  }) %>% bind_rows()
  
  return(paradise_df)
}

formatted_df <- get_paradise_output(all_sample_reps, celltype1, celltype2)
print("Finished formatting data")
print(output_filename)
# write_tsv(formatted_df, file = output_filename)


[1] "Celltype 1:  CANCERspecific-CH3-1"
[1] "Celltype 2:  CANCERspecific-FUBP1"
[1] "Output filename:  /mnt/dawnccle2/processed_data/reprocess_250221/pairadise_denovo_CANCER_PSI/WT_CANCERspecific-CH3-1_CANCERspecific-FUBP1/CANCERspecific-CH3-1_CANCERspecific-FUBP1_formatted_df.tsv"


[1] "Finished reading data"
[1] "CANCERspecific-CH3-1-A1-rep1" "CANCERspecific-CH3-1-A1-rep2"
[3] "CANCERspecific-CH3-1-A2-rep1" "CANCERspecific-CH3-1-A2-rep2"
[1] "CANCERspecific-FUBP1-B12-rep1" "CANCERspecific-FUBP1-B12-rep2"
[3] "CANCERspecific-FUBP1-C8-rep1"  "CANCERspecific-FUBP1-C8-rep2" 
[1] "Finished formatting data"
[1] "/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_denovo_CANCER_PSI/WT_CANCERspecific-CH3-1_CANCERspecific-FUBP1/CANCERspecific-CH3-1_CANCERspecific-FUBP1_formatted_df.tsv"


In [15]:
formatted_df

ExonID,I1,S1,I2,S2,I_len,S_len
<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>
ENSG00000006025.12;OSBPL7;chr17-47808863-47808946-47808537-47808660-47809075-47809115__0:0:0__R2design0,"10,22,13","54,64,36","9,9,15","37,65,33",1,1
ENSG00000006025.12;OSBPL7;chr17-47808863-47808946-47808537-47808660-47809075-47809115__0:0:0__vae_cancer_U2AF1S34F_0_to_neg_R2design1,"719,795,426,211","692,720,1033,602","449,714,343,138","643,823,754,309",1,1
ENSG00000006025.12;OSBPL7;chr17-47808863-47808946-47808537-47808660-47809075-47809115__0:0:0__vae_cancer_U2AF1S34F_0_to_neg_R2design3,"0,0,0","32,48,35","0,0","34,34",1,1
ENSG00000006025.12;OSBPL7;chr17-47808863-47808946-47808537-47808660-47809075-47809115__0:0:0__vae_cancer_U2AF1S34F_0_to_neg_R2design4,"64,56,28,14","515,449,561,434","47,59,9,15","418,573,416,179",1,1
ENSG00000006025.12;OSBPL7;chr17-47808863-47808946-47808537-47808660-47809075-47809115__0:0:0__vae_cancer_U2AF1S34F_0_to_neg_R2design5,"1,0,2,0","77,64,97,58","1,0,0,0","48,143,50,38",1,1
ENSG00000006025.12;OSBPL7;chr17-47808863-47808946-47808537-47808660-47809075-47809115__0:0:0__vae_cancer_U2AF1S34F_0_to_neg_R2design6,"4,0,3,2","79,76,109,96","4,4,0","83,117,68",1,1
ENSG00000006025.12;OSBPL7;chr17-47808863-47808946-47808537-47808660-47809075-47809115__0:0:0__vae_cancer_U2AF1S34F_0_to_neg_R2design7,"77,97,45,40","255,236,296,195","60,119,22,27","222,308,236,80",1,1
ENSG00000006025.12;OSBPL7;chr17-47808863-47808946-47808537-47808660-47809075-47809115__0:0:0__vae_cancer_U2AF1S34F_0_to_neg_R2design8,"0,0,0","53,44,44","0,0","31,30",1,1
ENSG00000006611.17;USH1C;chr11-17516240-17516290-17501938-17501980-17517400-17517474__0:0:0__R2design0,"0,2,1,1","1899,2055,2044,1180","5,28,6,0","1380,1987,1760,772",1,1
